# Chapter 71: Phishing Detection and Analysis

**Volume 4 — Security Operations**

**Phishing is the initial access vector in 80%+ of enterprise compromises.** After three decades
of security awareness training, DMARC, sandboxing, and multi-scanning email gateways —
attackers still get through. Not because the technical defenses are weak,
but because the attack is cognitive: urgency bypasses analysis, authority triggers compliance.
**Detection must happen before the email reaches the human.**

### What You Will Build — 5 Detection Demos

| Demo | Technique | What It Catches |
|------|-----------|----------------|
| **1. URL Lexical Feature Extractor** | Entropy + TLD risk + subdomain depth | Suspicious URL structure before any HTTP connection |
| **2. Email Header Analyzer** | SPF/DKIM/DMARC simulation + spoofing indicators | Display-name spoofing, envelope mismatch, first-contact risk |
| **3. Brand Impersonation Detector** | Levenshtein distance + homograph scoring | Lookalike domains impersonating known brands |
| **4. Link Risk Scorer from Email Body** | URL extraction + multi-signal risk scoring | Malicious links embedded in email body text |
| **5. Full Phishing Triage Pipeline** | All signals + LLM zero-shot classification | End-to-end verdict + response recommendation |

### The Core Insight
> **SPF, DKIM, and DMARC are like RPKI — they prevent exact-domain spoofing
> the same way Route Origin Authorization prevents exact-prefix hijacking.
> But they cannot stop a lookalike domain attack, just like RPKI cannot stop
> ASN-level social engineering. AI detection is the layer that catches what
> the technical validation misses: semantic intent, behavioral anomaly, structural deception.**

In [ ]:
# pip install anthropic
# Phishing Detection and Analysis — pure Python, no ML libraries required!

import os, json, re, math, time, random, hashlib
from dataclasses import dataclass, field
from typing import List, Dict, Optional
from collections import defaultdict

# ── Anthropic client ──────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    import getpass
    if not os.environ.get('ANTHROPIC_API_KEY'):
        os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Enter your Anthropic API key: ')

api_key = os.environ.get('ANTHROPIC_API_KEY', '')
USE_API = bool(api_key)

if USE_API:
    from anthropic import Anthropic
    client = Anthropic()
    print("LLM analyst: ACTIVE (Anthropic API)")
else:
    print("LLM analyst: SIMULATION MODE (set ANTHROPIC_API_KEY for real analysis)")

def llm_analyze(prompt: str, max_tokens: int = 200) -> str:
    if USE_API:
        resp = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}]
        )
        return resp.content[0].text.strip()
    # Simulation fallback — domain-specific canned responses
    if "bec" in prompt.lower() or "wire transfer" in prompt.lower() or "invoice" in prompt.lower():
        return ("VERDICT: PHISHING | CONFIDENCE: HIGH | "
                "KEY SIGNALS: First-contact sender, financial request, urgency language, lookalike domain | "
                "ACTION: Quarantine immediately, notify security team, do not process payment.")
    if "spf=fail" in prompt.lower() or "dmarc=fail" in prompt.lower():
        return ("VERDICT: PHISHING | CONFIDENCE: HIGH | "
                "KEY SIGNALS: Authentication failures, display-name spoofing | "
                "ACTION: Reject and quarantine. Block sending IP at email gateway.")
    if "entropy" in prompt.lower() or "lookalike" in prompt.lower():
        return ("VERDICT: SUSPICIOUS | CONFIDENCE: MEDIUM | "
                "KEY SIGNALS: High URL entropy, lookalike domain detected | "
                "ACTION: Sandbox URL, hold email pending analyst review.")
    return "VERDICT: LEGITIMATE | CONFIDENCE: HIGH | KEY SIGNALS: Auth passes, known sender | ACTION: Deliver."

print("Setup complete. All 5 phishing detection demos ready.")

## Demo 1: URL Lexical Feature Extractor

URL analysis is your first detection opportunity — before any HTTP connection is made,
before any reputation database is queried, the URL structure itself carries phishing signals.

**Five lexical features that distinguish phishing URLs from legitimate ones:**

| Feature | Why It Matters | Phishing Signal |
|---------|---------------|----------------|
| **Shannon Entropy** | Random characters score higher than dictionary words | > 3.8 bits/char in domain label |
| **Subdomain Depth** | Phishers nest legitimate brands as deep subdomains | depth ≥ 3 (e.g., paypal.com.attacker.io) |
| **TLD Risk Score** | Some TLDs are statistically over-represented in phishing | .tk, .ml, .ga, .cf, .gq free TLDs |
| **URL Length** | Legitimate URLs are short; phishing URLs hide behind length | > 75 characters total URL |
| **Special Character Count** | @, -, _ used to create visual confusion | Multiple hyphens, @ in path |

These checks complete in **microseconds** — run on every inbound email
before any heavyweight analysis is triggered.

*Network analogy: lexical URL analysis is like checking a BGP route announcement
for prefix length anomalies before even looking at path attributes.
A /32 announcement from an ASN that historically announces /24s is suspicious
on structure alone — no path inspection needed.*

In [ ]:
# ── Demo 1: URL Lexical Feature Extractor ─────────────────────────────────────

# TLD risk scores — higher = more frequently associated with phishing
# Source: APWG and SpamHaus statistical analysis
TLD_RISK = {
    ".tk": 0.95, ".ml": 0.92, ".ga": 0.90, ".cf": 0.90, ".gq": 0.89,  # Freenom free TLDs
    ".ru": 0.65, ".cn": 0.60, ".top": 0.58, ".xyz": 0.55, ".click": 0.70,
    ".work": 0.52, ".date": 0.68, ".download": 0.72, ".stream": 0.65,
    ".com": 0.10, ".net": 0.12, ".org": 0.08, ".gov": 0.02, ".edu": 0.02,
    ".co.uk": 0.10, ".de": 0.12, ".fr": 0.12, ".io": 0.20,
}

@dataclass
class URLFeatures:
    """Lexical features extracted from a URL — no HTTP connection required."""
    url: str
    scheme: str
    domain: str
    path: str
    tld: str
    subdomain_depth: int
    registered_label: str   # second-level domain label
    url_length: int
    entropy: float
    tld_risk: float
    hyphen_count: int
    at_in_url: bool
    ip_address: bool
    special_char_count: int
    phishing_score: float
    risk_factors: List[str]


def shannon_entropy(s: str) -> float:
    """Shannon entropy in bits per character. Higher = more random."""
    if not s:
        return 0.0
    freq: Dict[str, int] = defaultdict(int)
    for c in s:
        freq[c] += 1
    n = len(s)
    return -sum((c / n) * math.log2(c / n) for c in freq.values())


def extract_url_features(url: str) -> URLFeatures:
    """Extract all lexical features from a URL string — pure text, no network calls."""
    url_lower = url.lower().strip()

    # Scheme
    scheme_match = re.match(r'^(https?)://', url_lower)
    scheme = scheme_match.group(1) if scheme_match else "unknown"
    without_scheme = re.sub(r'^https?://', '', url_lower)

    # Domain vs path split
    path_split = without_scheme.split('/', 1)
    domain_part = path_split[0]
    path = '/' + path_split[1] if len(path_split) > 1 else '/'

    # IP address check (no domain label)
    ip_address = bool(re.match(r'^\d{1,3}(\.\d{1,3}){3}', domain_part))

    # Subdomain decomposition
    parts = domain_part.split('.')
    if len(parts) >= 2:
        tld = '.' + parts[-1]
        registered_label = parts[-2]  # second-level domain
        subdomains = parts[:-2]
        subdomain_depth = len(subdomains)
    else:
        tld = ''
        registered_label = domain_part
        subdomains = []
        subdomain_depth = 0

    # Entropy of the registered domain label
    entropy = shannon_entropy(registered_label)

    # TLD risk lookup
    tld_risk = TLD_RISK.get(tld, 0.25)  # default 0.25 for unknown TLDs

    # Special characters
    hyphen_count = url_lower.count('-')
    at_in_url = '@' in url_lower
    special_char_count = sum(url_lower.count(c) for c in ['-', '_', '~', '%', '@', '!'])

    # Build risk factor list and composite score
    risk_factors = []
    score_components = []

    # Entropy signal (weight 0.25)
    if entropy > 3.8:
        risk_factors.append(f"High domain entropy ({entropy:.2f} bits — algorithmically random)")
        score_components.append(min((entropy - 2.5) / 1.5, 1.0) * 0.25)
    else:
        score_components.append(0.0)

    # TLD risk (weight 0.30)
    if tld_risk >= 0.50:
        risk_factors.append(f"High-risk TLD: {tld} (score {tld_risk:.2f})")
    score_components.append(tld_risk * 0.30)

    # Subdomain depth (weight 0.20)
    if subdomain_depth >= 3:
        risk_factors.append(f"Deep subdomain nesting ({subdomain_depth} levels — brand abuse pattern)")
        score_components.append(min(subdomain_depth / 5.0, 1.0) * 0.20)
    elif subdomain_depth >= 2:
        score_components.append(0.10)
    else:
        score_components.append(0.0)

    # URL length (weight 0.10)
    if len(url) > 100:
        risk_factors.append(f"Excessive URL length ({len(url)} chars — hiding destination)")
        score_components.append(0.10)
    elif len(url) > 75:
        score_components.append(0.05)
    else:
        score_components.append(0.0)

    # Special characters (weight 0.10)
    if at_in_url:
        risk_factors.append("@ symbol in URL — browser ignores everything before @ as credentials")
        score_components.append(0.10)
    elif hyphen_count >= 4:
        risk_factors.append(f"Excessive hyphens ({hyphen_count}) — visual confusion tactic")
        score_components.append(0.06)
    else:
        score_components.append(0.0)

    # IP address instead of domain (weight 0.05)
    if ip_address:
        risk_factors.append("IP address used instead of domain — evading domain reputation")
        score_components.append(0.05)
    else:
        score_components.append(0.0)

    phishing_score = round(min(sum(score_components), 1.0), 3)

    return URLFeatures(
        url=url,
        scheme=scheme,
        domain=domain_part,
        path=path,
        tld=tld,
        subdomain_depth=subdomain_depth,
        registered_label=registered_label,
        url_length=len(url),
        entropy=round(entropy, 3),
        tld_risk=tld_risk,
        hyphen_count=hyphen_count,
        at_in_url=at_in_url,
        ip_address=ip_address,
        special_char_count=special_char_count,
        phishing_score=phishing_score,
        risk_factors=risk_factors,
    )


# ── Test URLs ─────────────────────────────────────────────────────────────────
print("=" * 70)
print("URL LEXICAL FEATURE EXTRACTOR")
print("=" * 70)

test_urls = [
    # Legitimate
    ("https://www.microsoft.com/en-us/security",          "legit Microsoft"),
    ("https://accounts.google.com/signin",                "legit Google"),
    ("https://github.com/anthropics/anthropic-sdk-python","legit GitHub"),
    # Phishing patterns
    ("https://paypal.com.verify-account.tk/login",         "brand in subdomain + .tk TLD"),
    ("https://secure-login-microsoft.com-account-verify.ru/signin",
                                                            "lookalike + .ru + hyphens"),
    ("https://185.220.101.47/payload.exe",                 "IP instead of domain"),
    ("https://legitimate-company@phishing-domain.ml/steal","@ confusion attack"),
    ("https://xkqpzbmvowf.xyz/c2/beacon",                  "DGA-style + .xyz"),
    ("https://amazon.com.aws-account-verify.secure-login.support.click/verify",
                                                            "deep subdomain brand abuse"),
]

print(f"\n{'URL (truncated)':<55} {'Score':>6} {'Entropy':>8} {'TLD Risk':>9} {'Verdict'}")
print("-" * 90)

flagged_urls = []
for url, label in test_urls:
    f = extract_url_features(url)
    verdict = "PHISHING" if f.phishing_score >= 0.50 else ("SUSPICIOUS" if f.phishing_score >= 0.30 else "CLEAN")
    url_short = url[:52] + "..." if len(url) > 55 else url
    flag = " <--" if verdict == "PHISHING" else ""
    print(f"{url_short:<55} {f.phishing_score:>6.3f} {f.entropy:>8.3f} {f.tld_risk:>9.2f} {verdict}{flag}")

    if verdict in ("PHISHING", "SUSPICIOUS"):
        flagged_urls.append(f)

print(f"\n[URL Extractor] {len(flagged_urls)} URLs flagged for deeper analysis")
print("\nTop risk factors from flagged URLs:")
for f in flagged_urls[:3]:
    print(f"  {f.domain}:")
    for rf in f.risk_factors:
        print(f"    - {rf}")

## Demo 2: Email Header Analyzer

Email headers contain the technical authentication record of every message.
Understanding what SPF/DKIM/DMARC protect — and what they **do not** — is
the foundation of email security architecture.

**What the three authentication mechanisms protect and miss:**

| Mechanism | Protects Against | Does NOT Protect Against |
|-----------|-----------------|-------------------------|
| **SPF** | IP spoofing (unauthorized sending IPs) | Display-from spoofing (name visible to user) |
| **DKIM** | Message tampering in transit | Lookalike domain (attacker signs with own domain) |
| **DMARC** | Display-from spoofing (closes SPF gap) | Lookalike domains with their own valid DMARC |

**Critical gap**: an attacker who registers `yourcompany-security.com`, sets up valid
SPF/DKIM/DMARC for that domain, and sends email claiming to be your security team —
all three checks pass. The lookalike domain is the attack. AI detection catches this;
technical validation cannot.

**Spoofing indicators this demo extracts:**
- Display-from vs envelope-from mismatch
- First-contact sender with financial request
- Authentication failures with suspicious content
- Received-from IP geographic anomaly (headers tell you where the mail actually came from)
- Reply-to address different from from-address (common BEC tactic)

In [ ]:
# ── Demo 2: Email Header Analyzer ─────────────────────────────────────────────

@dataclass
class HeaderAnalysis:
    """Structured signals extracted from email headers."""
    message_id: str
    from_display_name: str
    from_display_addr: str
    from_envelope_addr: str
    reply_to: str
    subject: str
    received_ip: str
    spf: str             # pass | fail | softfail | neutral | none
    dkim: str            # pass | fail | none
    dmarc: str           # pass | fail | none
    display_envelope_match: bool
    reply_to_mismatch: bool
    is_first_contact: bool
    contains_financial_keywords: bool
    urgency_score: float  # 0.0 - 1.0 based on urgency language
    spoofing_indicators: List[str]
    header_risk_score: float


# Urgency keywords — weighted by phishing pretext severity
URGENCY_KEYWORDS = {
    "urgent": 0.3, "immediately": 0.3, "asap": 0.25, "right away": 0.25,
    "time sensitive": 0.35, "closing today": 0.4, "within 24 hours": 0.35,
    "do not delay": 0.3, "action required": 0.2, "respond now": 0.3,
    "final notice": 0.4, "account suspended": 0.45, "verify now": 0.4,
    "expires today": 0.45, "limited time": 0.25,
}

FINANCIAL_KEYWORDS = [
    "wire transfer", "bank account", "invoice", "payment",
    "gift card", "banking details", "routing number", "ach transfer",
    "direct deposit", "wire instructions", "purchase order",
]


def analyze_headers(
    raw_headers: str,
    body_preview: str,
    sender_history: List[str],
) -> HeaderAnalysis:
    """Extract and score spoofing indicators from raw email headers."""

    # Parse From: field — handles 'Display Name <email@domain>' and bare addresses
    from_match = re.search(
        r'From:\s*"?([^"<\n]+?)"?\s*<?([a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,})>?',
        raw_headers
    )
    display_name = from_match.group(1).strip() if from_match else "unknown"
    display_addr = from_match.group(2).strip().lower() if from_match else "unknown"

    # Envelope from (SMTP MAIL FROM — may differ from header From:)
    envelope_match = re.search(r'Return-Path:\s*<?([^>\n]+)>?', raw_headers)
    envelope_addr = envelope_match.group(1).strip().lower() if envelope_match else display_addr

    # Reply-To
    reply_match = re.search(r'Reply-To:\s*([^\n]+)', raw_headers)
    reply_to = reply_match.group(1).strip().lower() if reply_match else ""

    # Subject
    subject_match = re.search(r'Subject:\s*(.+)', raw_headers)
    subject = subject_match.group(1).strip() if subject_match else ""

    # Received-from IP (last hop — originating server)
    ip_match = re.search(r'Received:.*?\((\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3})\)', raw_headers)
    received_ip = ip_match.group(1) if ip_match else "unknown"

    # Message-ID
    msgid_match = re.search(r'Message-ID:\s*<([^>]+)>', raw_headers)
    message_id = msgid_match.group(1) if msgid_match else "unknown"

    # Authentication results (simplified parsing)
    auth_block = raw_headers.lower()
    spf  = "pass" if "spf=pass"  in auth_block else ("fail" if "spf=fail" in auth_block else "none")
    dkim = "pass" if "dkim=pass" in auth_block else ("fail" if "dkim=fail" in auth_block else "none")
    dmarc= "pass" if "dmarc=pass"in auth_block else ("fail" if "dmarc=fail" in auth_block else "none")

    # Structural checks
    display_envelope_match = (
        display_addr.split('@')[-1] == envelope_addr.split('@')[-1]
    ) if '@' in display_addr and '@' in envelope_addr else False

    reply_to_mismatch = bool(reply_to and reply_to != display_addr)
    is_first_contact = display_addr not in sender_history

    # Financial and urgency scanning
    combined_text = (subject + " " + body_preview).lower()
    contains_financial = any(kw in combined_text for kw in FINANCIAL_KEYWORDS)
    urgency_score = min(
        sum(weight for kw, weight in URGENCY_KEYWORDS.items() if kw in combined_text),
        1.0
    )

    # Collect spoofing indicators
    indicators = []
    if not display_envelope_match:
        indicators.append(
            f"Display domain ({display_addr.split('@')[-1]}) != Envelope domain "
            f"({envelope_addr.split('@')[-1]}) — classic display-name spoofing"
        )
    if reply_to_mismatch:
        indicators.append(f"Reply-To mismatch: responses go to {reply_to} not sender")
    if spf == "fail":
        indicators.append("SPF FAIL: sending IP not authorized for this domain")
    if dmarc == "fail":
        indicators.append("DMARC FAIL: message fails alignment — reject per policy")
    if is_first_contact and contains_financial:
        indicators.append("First-contact sender making financial request — BEC pattern")
    if urgency_score >= 0.5:
        indicators.append(f"High urgency language (score {urgency_score:.2f}) — pressure tactic")

    # Composite header risk score
    score = 0.0
    if spf == "fail":   score += 0.30
    if dmarc == "fail": score += 0.25
    if not display_envelope_match: score += 0.20
    if reply_to_mismatch: score += 0.15
    if is_first_contact and contains_financial: score += 0.25
    if urgency_score >= 0.5: score += urgency_score * 0.15

    return HeaderAnalysis(
        message_id=message_id,
        from_display_name=display_name,
        from_display_addr=display_addr,
        from_envelope_addr=envelope_addr,
        reply_to=reply_to,
        subject=subject,
        received_ip=received_ip,
        spf=spf, dkim=dkim, dmarc=dmarc,
        display_envelope_match=display_envelope_match,
        reply_to_mismatch=reply_to_mismatch,
        is_first_contact=is_first_contact,
        contains_financial_keywords=contains_financial,
        urgency_score=round(urgency_score, 2),
        spoofing_indicators=indicators,
        header_risk_score=round(min(score, 1.0), 3),
    )


# ── Test emails ───────────────────────────────────────────────────────────────
print("=" * 65)
print("EMAIL HEADER ANALYZER")
print("=" * 65)

test_emails = [
    (
        "BEC wire transfer fraud",
        """From: "Robert Mitchell - CFO" <rmitchell@acme-corp-finance.com>
Return-Path: <bounce@acme-corp-finance.com>
Reply-To: robert.mitchell.cfo@gmail.com
To: accounts@yourcompany.com
Subject: URGENT: Update wire transfer instructions
Message-ID: <bec001@acme-corp-finance.com>
Authentication-Results: spf=pass dkim=pass dmarc=pass
Received: from mail.acme-corp-finance.com (185.220.101.47)""",
        "Please process the wire transfer of $47,500 immediately. Time sensitive.",
        ["billing@acme-corp.com", "support@acme-corp.com"],  # known senders
    ),
    (
        "Display-name spoofing with SPF fail",
        """From: "IT Security Team" <noreply@yourcompany-security.tk>
Return-Path: <spammer@attacker-hosted.ml>
To: all-staff@yourcompany.com
Subject: ACTION REQUIRED: Verify your account expires today
Message-ID: <spoof001@attacker.ml>
Authentication-Results: spf=fail dkim=fail dmarc=fail
Received: from mail.attacker.ml (91.108.4.55)""",
        "Your account will be suspended. Verify now: http://yourcompany-verify.tk/login",
        ["it-security@yourcompany.com"],
    ),
    (
        "Legitimate vendor email",
        """From: "Cisco Licensing Team" <licensing@cisco.com>
Return-Path: <licensing-bounce@cisco.com>
To: network-admin@yourcompany.com
Subject: Your SmartNet renewal is due in 30 days
Message-ID: <renewal001@cisco.com>
Authentication-Results: spf=pass dkim=pass dmarc=pass
Received: from mx1.cisco.com (72.163.4.161)""",
        "Your SmartNet contract expires in 30 days. Please contact your account manager.",
        ["licensing@cisco.com", "support@cisco.com"],
    ),
]

for label, headers, body, history in test_emails:
    analysis = analyze_headers(headers, body, history)
    verdict = "PHISHING" if analysis.header_risk_score >= 0.50 else (
              "SUSPICIOUS" if analysis.header_risk_score >= 0.25 else "CLEAN")

    print(f"\n[{label}]")
    print(f"  Subject      : {analysis.subject}")
    print(f"  Display From : {analysis.from_display_name} <{analysis.from_display_addr}>")
    print(f"  Envelope From: {analysis.from_envelope_addr}")
    if analysis.reply_to:
        print(f"  Reply-To     : {analysis.reply_to}")
    print(f"  Auth         : SPF={analysis.spf}  DKIM={analysis.dkim}  DMARC={analysis.dmarc}")
    print(f"  First contact: {analysis.is_first_contact} | Financial: {analysis.contains_financial_keywords}")
    print(f"  Urgency score: {analysis.urgency_score}")
    print(f"  Risk score   : {analysis.header_risk_score}  =>  {verdict}")
    if analysis.spoofing_indicators:
        print("  Indicators:")
        for ind in analysis.spoofing_indicators:
            print(f"    [!] {ind}")

print("\n[Header Analyzer] Note: DMARC pass does not mean legitimate. "
      "Lookalike domains pass all three checks.")

## Demo 3: Brand Impersonation Detector

The most sophisticated phishing attacks do not fail authentication —
they pass all three checks using a **lookalike domain** that impersonates a trusted brand.

**Four lookalike attack variants:**

| Variant | Example | Detection Technique |
|---------|---------|-------------------|
| **Character substitution** | `paypa1.com` (digit 1 for letter l) | Levenshtein distance to known brands |
| **Subdomain brand abuse** | `paypal.com.attacker.io` | Brand name in subdomain, not registrant |
| **Homograph** | `аpple.com` (Cyrillic а) | Unicode normalization + edit distance |
| **TLD variation** | `yourcompany.net`, `yourcompany-login.com` | Fuzzy match against brand list |

**Levenshtein distance** is the core algorithm: minimum character edit operations
(insert, delete, substitute) to transform one string into another.
A domain with edit distance ≤ 2 from any known trusted brand domain is a candidate.

```
Levenshtein("microsoft", "micosoft")  = 1  (missing 'r')
Levenshtein("paypal",    "paypa1")    = 1  (substitute l->1)
Levenshtein("amazon",   "arnazon")   = 1  (rn looks like m)
```

*Network analogy: brand impersonation detection is prefix hijacking detection.
You know 203.0.113.0/24 belongs to AS65001. If AS65099 announces 203.0.113.0/25,
the prefix is structurally similar enough to attract traffic — same as a lookalike domain.*

In [ ]:
# ── Demo 3: Brand Impersonation Detector ──────────────────────────────────────

# Known trusted brand domains — in production: pulled from threat intel + internal allowlist
TRUSTED_BRANDS = {
    "microsoft":   ["microsoft.com", "microsoftonline.com", "office.com", "azure.com"],
    "google":      ["google.com", "googleapis.com", "gmail.com", "goog.le"],
    "paypal":      ["paypal.com", "paypal.me"],
    "amazon":      ["amazon.com", "aws.amazon.com", "amazon.co.uk", "amazonaws.com"],
    "apple":       ["apple.com", "icloud.com", "appleid.apple.com"],
    "docusign":    ["docusign.com", "docusign.net"],
    "fedex":       ["fedex.com", "fedex.com.au"],
    "linkedin":    ["linkedin.com", "lnkd.in"],
    "dropbox":     ["dropbox.com", "dropboxusercontent.com"],
    "cisco":       ["cisco.com", "webex.com", "meraki.com"],
    "yourcompany": ["yourcompany.com", "yourcompany.io"],  # protect own domain
}

# Flatten to label-only list for edit distance comparisons
BRAND_LABELS: Dict[str, str] = {}  # label -> brand name
for brand, domains in TRUSTED_BRANDS.items():
    for d in domains:
        label = d.split('.')[0]  # extract second-level label
        BRAND_LABELS[label] = brand


def levenshtein(s1: str, s2: str) -> int:
    """Compute edit distance between two strings — O(m*n)."""
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if s1[i-1] == s2[j-1] else 1
            dp[i][j] = min(
                dp[i-1][j] + 1,       # deletion
                dp[i][j-1] + 1,       # insertion
                dp[i-1][j-1] + cost,  # substitution
            )
    return dp[m][n]


def normalize_homograph(label: str) -> str:
    """
    Normalize Unicode homographs to their Latin equivalents.
    Catches Cyrillic/Greek characters visually identical to Latin.
    Simplified mapping — production uses unicodedata.normalize + confusables DB.
    """
    homograph_map = {
        'а': 'a', 'е': 'e', 'о': 'o', 'р': 'p', 'с': 'c',  # Cyrillic
        'υ': 'u', 'ν': 'v', 'ο': 'o', 'ρ': 'p', 'α': 'a',  # Greek
        '0': 'o', '1': 'l', '3': 'e', '4': 'a', '5': 's',  # digit substitutions
        '|': 'l', 'vv': 'w',
    }
    result = label
    for fake, real in homograph_map.items():
        result = result.replace(fake, real)
    return result


@dataclass
class ImpersonationResult:
    domain: str
    registered_label: str
    normalized_label: str
    closest_brand: str
    closest_brand_label: str
    edit_distance: int
    impersonation_score: float   # 0.0=legit, 1.0=clear impersonation
    technique: str               # character_sub | subdomain_abuse | homograph | legitimate
    brand_in_subdomain: bool


def detect_impersonation(domain: str) -> ImpersonationResult:
    """Score a domain for brand impersonation using edit distance and normalization."""
    parts = domain.lower().rstrip('.').split('.')
    registered_label = parts[-2] if len(parts) >= 2 else domain
    subdomains = parts[:-2]

    # Normalize homographs before distance computation
    normalized = normalize_homograph(registered_label)

    # Check if any brand label appears in the subdomain chain
    brand_in_sub = False
    for brand_label in BRAND_LABELS:
        if any(brand_label in sub for sub in subdomains):
            brand_in_sub = True
            break

    # Find closest brand by edit distance
    best_brand = ""
    best_brand_label = ""
    best_dist = 999

    for brand_label, brand_name in BRAND_LABELS.items():
        # Skip very short labels to avoid false positives
        if len(brand_label) < 4:
            continue
        # Check against both original and normalized label
        dist = min(
            levenshtein(normalized, brand_label),
            levenshtein(registered_label, brand_label),
        )
        if dist < best_dist:
            best_dist = dist
            best_brand = brand_name
            best_brand_label = brand_label

    # Score and classify technique
    if registered_label in BRAND_LABELS:  # exact match = legitimate
        score = 0.0
        technique = "legitimate"
    elif brand_in_sub:
        score = 0.85
        technique = "subdomain_abuse"
    elif normalized != registered_label and best_dist <= 1:
        score = 0.90
        technique = "homograph"
    elif best_dist == 0:
        score = 0.0
        technique = "legitimate"
    elif best_dist == 1:
        score = 0.80
        technique = "character_substitution"
    elif best_dist == 2:
        score = 0.55
        technique = "close_lookalike"
    elif best_dist == 3:
        score = 0.30
        technique = "distant_lookalike"
    else:
        score = 0.0
        technique = "no_match"

    # Reduce score if exact brand is in the registered label but appended
    for brand_label in BRAND_LABELS:
        if brand_label in registered_label and registered_label != brand_label:
            score = max(score, 0.70)
            technique = "brand_augmentation"
            best_brand = BRAND_LABELS[brand_label]
            best_brand_label = brand_label
            best_dist = 0
            break

    return ImpersonationResult(
        domain=domain,
        registered_label=registered_label,
        normalized_label=normalized,
        closest_brand=best_brand,
        closest_brand_label=best_brand_label,
        edit_distance=best_dist,
        impersonation_score=round(score, 3),
        technique=technique,
        brand_in_subdomain=brand_in_sub,
    )


# ── Test domains ──────────────────────────────────────────────────────────────
print("=" * 75)
print("BRAND IMPERSONATION DETECTOR")
print("=" * 75)

test_domains = [
    # Legitimate brand domains
    "microsoft.com", "paypal.com", "amazon.com",
    # Character substitution attacks
    "paypa1.com",        # digit 1 for l
    "arnazon.com",       # rn looks like m
    "micosoft.com",      # missing r
    # Brand augmentation
    "paypal-verify.com",
    "microsoft-security-alert.com",
    "amazon-account-update.net",
    # Subdomain brand abuse
    "paypal.com.secure-login.tk",
    "apple.com.id-verify.ru",
    # Unrelated domains
    "cloudflare.com",
    "xkqpzbmvow.net",
]

print(f"{'Domain':<45} {'Score':>6} {'Edit':>5} {'Technique':<22} {'Impersonates'}")
print("-" * 95)

impersonation_hits = []
for domain in test_domains:
    result = detect_impersonation(domain)
    flag = " <--" if result.impersonation_score >= 0.55 else ""
    brand_str = result.closest_brand if result.impersonation_score >= 0.55 else "-"
    print(
        f"{domain:<45} {result.impersonation_score:>6.3f} "
        f"{result.edit_distance:>5} {result.technique:<22} {brand_str}{flag}"
    )
    if result.impersonation_score >= 0.55:
        impersonation_hits.append(result)

# LLM enrichment for top hit
if impersonation_hits:
    top = max(impersonation_hits, key=lambda r: r.impersonation_score)
    analysis = llm_analyze(
        f"Brand impersonation detected: domain '{top.domain}' scored "
        f"{top.impersonation_score:.2f} impersonation score targeting '{top.closest_brand}'. "
        f"Technique: {top.technique}. "
        f"What immediate action should the security team take? Under 60 words.",
        max_tokens=80
    )
    print(f"\n[LLM] Top impersonation hit ({top.domain}):")
    print(f"  {analysis}")

print(f"\n[Impersonation Detector] {len(impersonation_hits)} impersonation attempts detected.")

## Demo 4: Link Risk Scorer from Email Body

Every URL in an email body is a potential payload delivery vector.
This demo extracts all links from an email and scores each one using
multiple independent signals — combining lexical features, brand impersonation,
and contextual risk from the surrounding text.

**Multi-signal URL risk scoring pipeline:**

```
Email body text
    |
    v
[URL Extractor]      -> finds all http/https/shortened URLs
    |
    v
[Lexical Scorer]     -> entropy + TLD risk + length + special chars
    |
    v
[Impersonation Check]-> Levenshtein distance to known brand domains
    |
    v
[Context Scorer]     -> is link surrounded by urgency/financial keywords?
    |
    v
[Combined Risk Score]-> weighted aggregate (0.0 = clean, 1.0 = block)
```

**URL shortener detection** matters separately: `bit.ly`, `tinyurl.com`,
`t.co`, and `goo.gl` hide the real destination. A legitimate marketing email
may use a shortener. A phishing email definitely might. The shortener itself
is a risk factor — unresolved destination = cannot score the actual URL.

*Production note: resolve shortened URLs via HEAD request before scoring.
Never follow redirects to the final destination — sandbox it.*

In [ ]:
# ── Demo 4: Link Risk Scorer from Email Body ───────────────────────────────────

# Known URL shorteners — real destination hidden, always flag for sandbox resolution
URL_SHORTENERS = {
    "bit.ly", "tinyurl.com", "t.co", "goo.gl", "ow.ly", "is.gd",
    "buff.ly", "adf.ly", "shorturl.at", "cutt.ly", "rb.gy",
}

# Context risk keywords — when found near a URL, elevate risk
CONTEXT_RISK_PHRASES = {
    "click here": 0.20, "click the link": 0.20, "click below": 0.15,
    "verify your": 0.30, "confirm your": 0.30, "update your": 0.25,
    "account suspended": 0.40, "account locked": 0.40, "unusual activity": 0.35,
    "expires today": 0.40, "immediately": 0.25, "urgent": 0.25,
    "wire transfer": 0.45, "gift card": 0.45, "invoice attached": 0.35,
    "reset your password": 0.30, "sign in": 0.15,
}


@dataclass
class LinkRiskResult:
    url: str
    is_shortener: bool
    lexical_score: float
    impersonation_score: float
    context_score: float
    combined_risk: float
    risk_level: str   # CRITICAL | HIGH | MEDIUM | LOW | CLEAN
    risk_reasons: List[str]


def extract_urls(text: str) -> List[str]:
    """Extract all URLs from email body text."""
    # Match http/https URLs and common shortened formats
    pattern = r'https?://[^\s<>"\)\]]+|(?:bit\.ly|tinyurl\.com|t\.co)/\S+'
    return re.findall(pattern, text, re.IGNORECASE)


def score_url_context(url: str, body: str) -> float:
    """
    Score the risk context surrounding a URL in the email body.
    Extracts a 100-character window around each URL occurrence.
    """
    body_lower = body.lower()
    url_lower = url.lower()
    url_pos = body_lower.find(url_lower[:30])  # find by first 30 chars

    if url_pos == -1:
        # URL not found literally (may be in anchor text) — check full body
        context = body_lower
    else:
        # Extract 150-char window around URL
        start = max(0, url_pos - 150)
        end   = min(len(body_lower), url_pos + len(url) + 150)
        context = body_lower[start:end]

    context_score = min(
        sum(weight for phrase, weight in CONTEXT_RISK_PHRASES.items() if phrase in context),
        1.0
    )
    return round(context_score, 3)


def score_link(url: str, body: str) -> LinkRiskResult:
    """Multi-signal risk score for a single URL from an email body."""
    reasons = []

    # Extract domain for shortener check
    domain_match = re.search(r'https?://([^/]+)', url)
    domain = domain_match.group(1).lower() if domain_match else url

    # Shortener check
    is_shortener = domain in URL_SHORTENERS
    if is_shortener:
        reasons.append(f"URL shortener ({domain}) — real destination hidden, sandbox required")

    # Lexical scoring (Demo 1)
    lex_features = extract_url_features(url)
    lexical_score = lex_features.phishing_score
    reasons.extend(lex_features.risk_factors)

    # Brand impersonation scoring (Demo 3)
    imp_result = detect_impersonation(domain)
    impersonation_score = imp_result.impersonation_score
    if impersonation_score >= 0.55:
        reasons.append(
            f"Impersonates '{imp_result.closest_brand}' via {imp_result.technique} "
            f"(edit dist={imp_result.edit_distance})"
        )

    # Context score
    context_score = score_url_context(url, body)
    if context_score >= 0.30:
        reasons.append(f"High-risk context surrounding link (context score={context_score:.2f})")

    # Shortener adds flat risk
    shortener_score = 0.30 if is_shortener else 0.0

    # Weighted combination
    combined = min(
        lexical_score      * 0.35 +
        impersonation_score* 0.35 +
        context_score      * 0.20 +
        shortener_score    * 0.10,
        1.0
    )
    combined = round(combined, 3)

    if combined >= 0.75:   risk_level = "CRITICAL"
    elif combined >= 0.55: risk_level = "HIGH"
    elif combined >= 0.35: risk_level = "MEDIUM"
    elif combined >= 0.15: risk_level = "LOW"
    else:                  risk_level = "CLEAN"

    return LinkRiskResult(
        url=url,
        is_shortener=is_shortener,
        lexical_score=lexical_score,
        impersonation_score=impersonation_score,
        context_score=context_score,
        combined_risk=combined,
        risk_level=risk_level,
        risk_reasons=reasons,
    )


# ── Test with a realistic BEC phishing email ──────────────────────────────────
print("=" * 65)
print("LINK RISK SCORER FROM EMAIL BODY")
print("=" * 65)

phishing_body = """
Dear Valued Customer,

We have detected unusual activity on your PayPal account. Your account
has been temporarily suspended. Please verify your account immediately
to restore full access:

  https://paypa1.com.secure-verify-account.tk/login?user=you&session=x7k2m

Alternatively, you can confirm your details here:
  http://bit.ly/3xK9mRq

If you do not verify within 24 hours, your account will be permanently
suspended and all pending transactions will be cancelled.

For security, also review our updated banking details:
  https://microsoft.com.account-verify.ru/update-payment

This is time sensitive.

PayPal Security Team
https://paypal.com/help  <-- this one is actually legitimate
"""

urls = extract_urls(phishing_body)
print(f"\nExtracted {len(urls)} URLs from email body:")

results = []
for url in urls:
    result = score_link(url, phishing_body)
    results.append(result)

    print(f"\nURL: {url[:70]}{'...' if len(url) > 70 else ''}")
    print(f"  Lexical      : {result.lexical_score:.3f}")
    print(f"  Impersonation: {result.impersonation_score:.3f}")
    print(f"  Context      : {result.context_score:.3f}")
    print(f"  Combined Risk: {result.combined_risk:.3f}  =>  {result.risk_level}")
    if result.risk_reasons:
        print("  Reasons:")
        for r in result.risk_reasons[:3]:
            print(f"    [!] {r}")

critical = [r for r in results if r.risk_level in ("CRITICAL", "HIGH")]
print(f"\n[Link Scorer] {len(critical)}/{len(results)} links flagged CRITICAL or HIGH.")
print("In production: CRITICAL links trigger immediate quarantine + sandbox detonation.")

## Demo 5: Full Phishing Triage Pipeline

All four detection engines wired together into a complete phishing analysis pipeline,
the way a production AI-augmented email security system operates.

```
Inbound email
    |
    +-> Header Analyzer    -> authentication signals + spoofing indicators
    +-> URL Extractor      -> all links extracted from body
    |   +-> Lexical Scorer -> structure risk per URL
    |   +-> Impersonation  -> brand lookalike detection per URL
    |   +-> Context Scorer -> surrounding text risk
    |
    +-> Combined Signal Summary
    |
    v
[LLM Zero-Shot Classifier] <- receives structured signals, not raw email
    |
    v
VERDICT: PHISHING / SUSPICIOUS / LEGITIMATE
CONFIDENCE: HIGH / MEDIUM / LOW
RECOMMENDED ACTION: quarantine / sandbox / deliver / escalate
```

**The hybrid architecture advantage**: deterministic checks (SPF/DKIM/DMARC, lexical URL)
handle the clear-cut cases at high throughput and low cost.
The LLM handles the ambiguous middle: first-contact BEC, lookalike domains,
AI-generated phishing with perfect grammar.

> **Non-negotiable guardrail: LLM phishing verdicts require analyst confirmation
> before blocking or quarantine decisions that affect critical business communications.
> Automatically quarantine PHISHING+HIGH confidence. Escalate SUSPICIOUS to analyst.
> Never auto-delete — always quarantine with retrieval path.**

In [ ]:
# ── Demo 5: Full Phishing Triage Pipeline ─────────────────────────────────────

@dataclass
class PhishingVerdict:
    email_id: str
    verdict: str            # PHISHING | SUSPICIOUS | LEGITIMATE
    confidence: str         # HIGH | MEDIUM | LOW
    overall_risk_score: float
    header_risk: float
    max_url_risk: float
    llm_analysis: str
    recommended_action: str
    key_signals: List[str]
    processing_time_ms: float


class PhishingPipeline:
    """Full phishing detection pipeline: headers + URLs + LLM zero-shot classification."""

    def __init__(self):
        self.verdicts: List[PhishingVerdict] = []

    def analyze(
        self,
        email_id: str,
        raw_headers: str,
        body: str,
        sender_history: List[str],
    ) -> PhishingVerdict:
        """Run the full pipeline on one email. Returns a structured verdict."""
        start = time.time()

        # Stage 1: Header analysis
        header = analyze_headers(raw_headers, body, sender_history)

        # Stage 2: URL extraction and multi-signal scoring
        urls = extract_urls(body)
        url_results = [score_link(u, body) for u in urls]
        max_url_risk = max((r.combined_risk for r in url_results), default=0.0)

        # Stage 3: Aggregate signals
        all_signals = []
        all_signals.extend(header.spoofing_indicators)
        for ur in url_results:
            if ur.risk_level in ("CRITICAL", "HIGH"):
                all_signals.append(
                    f"URL risk={ur.combined_risk:.2f} ({ur.risk_level}): {ur.url[:50]}..."
                )
                all_signals.extend(ur.risk_reasons[:2])

        # Combined risk score: header + URL weighted
        combined_risk = round(
            header.header_risk_score * 0.45 +
            max_url_risk             * 0.40 +
            header.urgency_score     * 0.15,
            3
        )

        # Stage 4: LLM zero-shot classification
        # Feed structured signals — not raw email — to control prompt injection risk
        url_summary = (
            "None found" if not url_results else
            "; ".join(
                f"{r.url[:40]}... [risk={r.combined_risk:.2f}]"
                for r in url_results[:3]
            )
        )

        llm_prompt = (
            f"Analyze this email for phishing. Provide structured verdict.\n\n"
            f"HEADER SIGNALS:\n"
            f"- From display: {header.from_display_name} <{header.from_display_addr}>\n"
            f"- Envelope: {header.from_envelope_addr}\n"
            f"- Subject: {header.subject}\n"
            f"- SPF={header.spf} DKIM={header.dkim} DMARC={header.dmarc}\n"
            f"- First contact: {header.is_first_contact} | Financial request: {header.contains_financial_keywords}\n"
            f"- Urgency score: {header.urgency_score} | Header risk: {header.header_risk_score}\n"
            f"- Spoofing indicators: {'; '.join(header.spoofing_indicators) or 'None'}\n\n"
            f"URL SIGNALS (max risk={max_url_risk:.2f}):\n"
            f"- {url_summary}\n\n"
            f"EMAIL BODY PREVIEW (first 300 chars):\n"
            f"{body[:300]}\n\n"
            f"Respond in this EXACT format:\n"
            f"VERDICT: [PHISHING/SUSPICIOUS/LEGITIMATE]\n"
            f"CONFIDENCE: [HIGH/MEDIUM/LOW]\n"
            f"KEY SIGNALS: [3 bullet points, most important indicators]\n"
            f"ACTION: [one sentence — what email system should do]\n"
        )

        llm_response = llm_analyze(llm_prompt, max_tokens=220)

        # Parse verdict and confidence from LLM response
        verdict = "SUSPICIOUS"  # default
        confidence = "MEDIUM"
        if "VERDICT: PHISHING" in llm_response.upper():
            verdict = "PHISHING"
        elif "VERDICT: LEGITIMATE" in llm_response.upper():
            verdict = "LEGITIMATE"
        if "CONFIDENCE: HIGH" in llm_response.upper():
            confidence = "HIGH"
        elif "CONFIDENCE: LOW" in llm_response.upper():
            confidence = "LOW"

        # Determine recommended action
        if verdict == "PHISHING" and confidence == "HIGH":
            action = "AUTO-QUARANTINE: move to quarantine, notify sender's domain abuse contact"
        elif verdict == "PHISHING":
            action = "QUARANTINE + ANALYST REVIEW: hold for human confirmation before release"
        elif verdict == "SUSPICIOUS":
            action = "ANALYST ESCALATION: deliver with warning banner, flag for review"
        else:
            action = "DELIVER: pass to recipient with no restriction"

        elapsed_ms = (time.time() - start) * 1000

        result = PhishingVerdict(
            email_id=email_id,
            verdict=verdict,
            confidence=confidence,
            overall_risk_score=combined_risk,
            header_risk=header.header_risk_score,
            max_url_risk=max_url_risk,
            llm_analysis=llm_response,
            recommended_action=action,
            key_signals=all_signals[:5],
            processing_time_ms=round(elapsed_ms, 1),
        )
        self.verdicts.append(result)
        return result

    def print_verdict(self, v: PhishingVerdict):
        print(f"\n{'='*65}")
        print(f"PHISHING VERDICT — {v.email_id}")
        print(f"{'='*65}")
        print(f"  Verdict      : {v.verdict}")
        print(f"  Confidence   : {v.confidence}")
        print(f"  Risk Score   : {v.overall_risk_score:.3f} "
              f"(header={v.header_risk:.2f}, max_url={v.max_url_risk:.2f})")
        print(f"  Action       : {v.recommended_action}")
        print(f"  Processing   : {v.processing_time_ms:.0f}ms")
        print(f"\n  LLM Analysis:\n  {v.llm_analysis}")
        if v.key_signals:
            print(f"\n  Key signals detected:")
            for s in v.key_signals[:4]:
                print(f"    [!] {s}")


# ── Run the full pipeline on test emails ──────────────────────────────────────
pipeline = PhishingPipeline()

test_cases = [
    {
        "id": "MSG-2026-44821",
        "description": "BEC wire fraud — DMARC pass, lookalike domain",
        "headers": """From: \"Robert Mitchell - CFO\" <rmitchell@acme-corp-finance.com>
Return-Path: <bounce@acme-corp-finance.com>
Subject: URGENT: Update wire transfer instructions immediately
Authentication-Results: spf=pass dkim=pass dmarc=pass
Received: from mail.acme-corp-finance.com (185.220.101.47)""",
        "body": (
            "Hi, I need you to urgently update the wire transfer instructions for our Q4 payment. "
            "We changed banking providers. Please wire $47,500 to this new account immediately.\n"
            "Bank: First National | Account: 8847291047 | Routing: 021000021\n"
            "Confirm here: https://acme-corp-finance.com.wire-update.tk/confirm\n"
            "This is time sensitive — closing books for Q4."
        ),
        "history": ["billing@acme-corp.com", "support@acme-corp.com"],
    },
    {
        "id": "MSG-2026-44822",
        "description": "Credential harvest — SPF/DMARC fail, fake IT security",
        "headers": """From: \"IT Security Team\" <noreply@yourcompany-security.tk>
Return-Path: <spammer@attacker.ml>
Subject: ACTION REQUIRED: Your account expires today — verify now
Authentication-Results: spf=fail dkim=fail dmarc=fail
Received: from mail.attacker.ml (91.108.4.55)""",
        "body": (
            "Your account will be suspended in 24 hours due to unusual activity. "
            "Verify your credentials immediately to avoid losing access.\n"
            "Click here to verify: https://yourcompany-security.tk/verify?token=abc123\n"
            "Do not delay — account expires today."
        ),
        "history": ["it-security@yourcompany.com"],
    },
    {
        "id": "MSG-2026-44823",
        "description": "Legitimate vendor email — SmartNet renewal",
        "headers": """From: \"Cisco Licensing\" <licensing@cisco.com>
Return-Path: <licensing-bounce@cisco.com>
Subject: SmartNet renewal reminder — contract due in 30 days
Authentication-Results: spf=pass dkim=pass dmarc=pass
Received: from mx1.cisco.com (72.163.4.161)""",
        "body": (
            "Your SmartNet contract for Catalyst 9300 series expires in 30 days. "
            "Please contact your Cisco account manager or visit:\n"
            "https://cisco.com/c/en/us/services/smart-net-total-care.html\n"
            "No action required if you have already renewed."
        ),
        "history": ["licensing@cisco.com", "support@cisco.com"],
    },
]

for tc in test_cases:
    print(f"\n--- Analyzing: {tc['description']} ---")
    verdict = pipeline.analyze(tc["id"], tc["headers"], tc["body"], tc["history"])
    pipeline.print_verdict(verdict)

# Summary
print(f"\n{'='*65}")
print("PIPELINE SUMMARY")
print(f"{'='*65}")
for v in pipeline.verdicts:
    print(f"  {v.email_id}  {v.verdict:<12} {v.confidence:<8} score={v.overall_risk_score:.3f}")
print()
print("Non-negotiable: SUSPICIOUS verdicts require analyst confirmation before quarantine.")
print("Never auto-delete — always quarantine with analyst retrieval path.")

## Summary: What You Built

You now have a working **Phishing Detection and Analysis** pipeline with 5 detection engines:

| Engine | Technique | Key Metric | Production Threshold |
|--------|-----------|------------|---------------------|
| **URL Lexical Extractor** | Entropy + TLD risk + subdomain depth | Phishing score | > 0.50 = flag |
| **Header Analyzer** | SPF/DKIM/DMARC + spoofing indicators | Header risk score | > 0.50 = suspicious |
| **Brand Impersonation** | Levenshtein distance + homograph normalization | Impersonation score | ≤ 2 edit dist = flag |
| **Link Risk Scorer** | Lexical + impersonation + context weighted | Combined link risk | > 0.55 = HIGH |
| **Full Pipeline** | All engines + LLM zero-shot | Structured verdict | PHISHING+HIGH = quarantine |

### Why Grammar-Based Detection Is Dead

LLM-generated phishing is grammatically perfect. The detection signal has shifted:
- **Dead signal**: poor grammar, unusual phrasing
- **Live signal**: first-contact + financial request, behavioral deviation from communication history
- **Live signal**: lookalike domain structure (Levenshtein ≤ 2 from known brand)
- **Live signal**: SPF/DKIM/DMARC misalignment (envelope vs display domain)

### The Non-Negotiable Rule

> **Quarantine is reversible. Deletion is not.
> Auto-quarantine PHISHING+HIGH confidence. Escalate SUSPICIOUS to analyst.
> Never auto-delete — always quarantine with a retrieval path for false positives.
> A legitimate email blocked causes immediate visible business impact;
> that visibility is what keeps false positive rates accountable.**

### Production Upgrade Path

```
Regex email parser          ->  Python email.parser module + MIME handling
Simulated SPF/DKIM checks   ->  pyspf, dkimpy, py-dmarc libraries
Static TLD risk table       ->  APWG eCrime threat intel feed (live updates)
Levenshtein from scratch    ->  python-Levenshtein (C extension, 10x faster)
URL shortener flag          ->  HEAD request expansion before scoring
claude-haiku-4-5-20251001   ->  claude-sonnet-4-5-20250514 for BEC analysis
In-memory verdict store     ->  SIEM integration for trend analysis
```

**Next: Chapter 72 — Security Log Analysis and SIEM Integration** —
the email verdicts from this chapter flow into SIEM correlation:
phishing delivered + outbound connection to URL IP + authentication with harvested credentials
= full attack chain reconstruction.

In [ ]:
# ── Production Deployment Checklist + Key Formulas ────────────────────────────
print("CHAPTER 71: PHISHING DETECTION — PRODUCTION CHECKLIST")
print("=" * 65)

checklist = [
    # Data Pipeline
    ("Data Pipeline",   "Email telemetry as first-class data stream (same mindset as NetFlow)"),
    ("Data Pipeline",   "DNS-first URL reputation: DNSBL/SURBL check before HTTP sandbox"),
    ("Data Pipeline",   "Registration age check: newly registered domains are high risk"),
    ("Data Pipeline",   "Network connection logs: did any host connect to flagged URL IP?"),
    # Authentication Checks
    ("Authentication",  "SPF: validates sending IP but NOT display-from address"),
    ("Authentication",  "DKIM: validates message integrity but NOT legitimacy of signing domain"),
    ("Authentication",  "DMARC: closes display-from gap but NOT lookalike domain attacks"),
    ("Authentication",  "DMARC pass + lookalike domain = phishing — AI layer required"),
    # Lookalike Detection
    ("Lookalike",       "Levenshtein distance <= 2 from any known brand label = flag"),
    ("Lookalike",       "Unicode normalization before edit distance (homograph attacks)"),
    ("Lookalike",       "Brand label in subdomain = subdomain abuse attack"),
    ("Lookalike",       "Known brand label + any suffix = brand augmentation attack"),
    # BEC Detection
    ("BEC Detection",   "First-contact sender with financial request = immediate escalation"),
    ("BEC Detection",   "Communication style baseline per sender pair (LLM style analysis)"),
    ("BEC Detection",   "Reply-To mismatch = responses go to attacker not sender"),
    ("BEC Detection",   "Timing anomaly: Friday PM / executive travel / quarter-end = elevated"),
    # Quarantine Policy
    ("Quarantine",      "PHISHING + HIGH confidence: auto-quarantine"),
    ("Quarantine",      "PHISHING + MEDIUM confidence: quarantine + analyst review"),
    ("Quarantine",      "SUSPICIOUS: deliver with warning banner + analyst flag"),
    ("Quarantine",      "NEVER auto-delete — always maintain retrieval path"),
    # LLM Integration
    ("LLM Integration", "Feed structured signals to LLM — not raw email (prompt injection risk)"),
    ("LLM Integration", "claude-haiku-4-5-20251001 for bulk triage (fast, cost-effective)"),
    ("LLM Integration", "claude-sonnet-4-5-20250514 for BEC and targeted spearphishing"),
    ("LLM Integration", "Require structured output format: VERDICT / CONFIDENCE / ACTION"),
]

current_cat = ""
for category, item in checklist:
    if category != current_cat:
        print(f"\n[{category}]")
        current_cat = category
    print(f"  + {item}")

print()
print("=" * 65)
print("KEY FORMULAS")
print("=" * 65)
print("URL lexical score   = 0.25*(entropy) + 0.30*(tld_risk) + 0.20*(subdomain) + 0.10*(length) + 0.10*(special) + 0.05*(ip)")
print("Impersonation score = edit_dist<=1 -> 0.80 | brand_in_subdomain -> 0.85 | brand+suffix -> 0.70")
print("Link combined risk  = 0.35*(lexical) + 0.35*(impersonation) + 0.20*(context) + 0.10*(shortener)")
print("Header risk score   = 0.30*(spf_fail) + 0.25*(dmarc_fail) + 0.20*(envelope_mismatch) + 0.15*(reply_to) + ...")
print("Pipeline score      = 0.45*(header_risk) + 0.40*(max_url_risk) + 0.15*(urgency_score)")
print()
print("Levenshtein distance threshold:")
print("  dist = 0 : exact brand match (legitimate)")
print("  dist = 1 : high confidence impersonation (score 0.80)")
print("  dist = 2 : moderate impersonation (score 0.55)")
print("  dist = 3 : possible lookalike (score 0.30) — escalate for human review")
print("  dist >= 4: no match")
print()
print("Chapter 71 complete. Email telemetry is a first-class network data stream.")
print("DMARC pass does not mean legitimate. AI is the layer that catches what rules miss.")